# Review treatment-responsive genes
Shift-drag to brush; click a gene to select it. Save a selection into a Python-owned shortlist. The browser version also supports row removal.


In [ ]:
import genome_spy as gs
from genome_spy.datasets import load_dataset

# The same prepared statistics, genes and sample counts as the gallery example.
data = load_dataset("airway_review", as_format="json")
# Same bound cutoff parameters and classification as the gallery volcano plot.
effect_cutoff = gs.param(
    "airwayVolcanoEffectCutoff",
    value=1.0,
    bind=gs.binding_range(
        min=0, max=3, step=0.1, name="Absolute log2 fold-change cutoff: "
    ),
)
significance_cutoff = gs.param(
    "airwayVolcanoSignificanceCutoff",
    value=data["domains"]["pvalue_cutoff"][0],
    bind=gs.binding_range(
        min=0, max=data["domains"]["volcano_y"][1], step=0.25, name="−log10 p cutoff: "
    ),
)
direction = gs.expr.if_(
    (gs.datum.neglog10_pvalue >= significance_cutoff)
    & (gs.expr.abs(gs.datum.log2fc) >= effect_cutoff),
    gs.expr.if_(gs.datum.log2fc < 0, "down in dex", "up in dex"),
    "n.s.",
)
brush = gs.selection_interval(
    "brush", encodings=["x", "y"], empty=False, on="mousedown[event.shiftKey]"
)
picked = gs.selection_point(
    "picked", empty=False, toggle=False, on="click[!event.shiftKey]"
)
points = (
    gs.Chart()
    .transform_collect()
    .transform_formula(expr=direction, as_="direction")
    .mark_point(
        size=gs.expr("min(14 * pow(zoomLevel(), 0.75), 64)"), opacity=0.58, filled=True
    )
    .encode(
        x=gs.X("log2fc:Q")
        .scale(domain=data["domains"]["volcano_x"], zoom=True, nice=False)
        .title("log2 fold change · dexamethasone / control"),
        y=gs.Y("neglog10_pvalue_plot:Q")
        .scale(domain=data["domains"]["volcano_y"], zoom=True, nice=False)
        .title("−log10 p-value"),
        color=gs.Color("direction:N").scale(
            domain=["down in dex", "n.s.", "up in dex"],
            range=["#3e8cb6", "#c9d1d9", "#c53b2c"],
        ),
        stroke=gs.value("#222222"),
        strokeOpacity=gs.when({"or": [picked, brush]})
        .then(gs.value(1))
        .otherwise(gs.value(0)),
        strokeWidth=gs.value(1.2),
        tooltip=["gene_id:N", "symbol:N", "log2fc:Q", "padj:Q", "baseMean:Q"],
    )
    .add_params(picked)
    .properties(name="volcano")
)
fc_rules = (
    gs.Chart([{"side": -1}, {"side": 1}])
    .transform_collect()
    .transform_formula(expr=gs.datum.side * effect_cutoff, as_="x")
    .mark_rule(strokeDash=[4, 4], size=1, color="#8f98a3", tooltip=None)
    .encode(
        x=gs.X("x:Q", title="log2 fold change · dexamethasone / control").scale(
            domain=data["domains"]["volcano_x"], zoom=True, nice=False
        )
    )
    .properties(name="volcano-fc-rules")
)
p_rule = (
    gs.Chart([{}])
    .transform_collect()
    .transform_formula(expr=significance_cutoff, as_="y")
    .mark_rule(strokeDash=[4, 4], size=1, color="#8f98a3", tooltip=None)
    .encode(
        y=gs.Y("y:Q", title="−log10 p-value").scale(
            domain=data["domains"]["volcano_y"], zoom=True, nice=False
        )
    )
    .properties(name="volcano-p-rule")
)
chart = (
    (fc_rules + p_rule + points)
    .add_params(brush, effect_cutoff, significance_cutoff)
    .properties(
        data=data["genes"], width=820, height=400, datasets={"samples": data["samples"]}
    )
)

In [ ]:
import asyncio
import html
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

if old_task := globals().get("connection_task"):
    old_task.cancel()
if old_widget := globals().get("widget"):
    old_widget.close()
widget = chart.widget(inline=True, controls=False)
status = widgets.HTML("Connecting…")
table = widgets.HTML()
shortlist_table = widgets.HTML()
note = widgets.Text(description="Note")
add = widgets.Button(description="Save selection", disabled=True)
selected_genes = pd.DataFrame()
shortlist = {}
selected_kind = None
columns = ["gene_id", "symbol", "log2fc", "padj"]


def show(rows):
    global selected_genes
    selected_genes = pd.DataFrame(rows)
    status.value = f"{len(rows)} genes selected; showing the first 20."
    table.value = (
        selected_genes[columns].head(20).to_html(index=False, escape=True)
        if rows
        else ""
    )
    add.disabled = not rows


def brush_changed(snapshot):
    global selected_kind
    if not snapshot["active"]:
        if selected_kind == "brush":
            show([])
        return
    selected_kind = "brush"
    asyncio.create_task(pick_handle.clear())
    x, y = snapshot["intervals"]["x"], snapshot["intervals"]["y"]
    show(
        [
            row
            for row in data["genes"]
            if min(x) <= row["log2fc"] <= max(x)
            and min(y) <= row["neglog10_pvalue_plot"] <= max(y)
        ]
    )


def point_changed(snapshot):
    global selected_kind
    if not snapshot["active"]:
        if selected_kind == "picked":
            show([])
        return
    selected_kind = "picked"
    asyncio.create_task(brush_handle.clear())
    show(snapshot["data"])


def save_selection(_):
    for row in selected_genes.to_dict("records"):
        shortlist.setdefault(row["gene_id"], {**row, "note": note.value})
    shortlist_table.value = pd.DataFrame(shortlist.values())[
        columns + ["note"]
    ].to_html(index=False, escape=True)


add.on_click(save_selection)


async def connect():
    global api, stops, brush_handle, pick_handle
    try:
        api = await widget.get_embed_api()
        brush_handle = await api.params.get_selection("brush")
        view = await api.views.get({"scope": [], "view": "volcano"})
        pick_handle = await view.params.get_selection("picked")
        stops = [
            await brush_handle.subscribe(brush_changed, options={"delivery": "commit"}),
            await pick_handle.subscribe(point_changed),
        ]
        status.value = "Shift-drag to brush; click a gene to select it."
    except Exception as error:
        status.value = html.escape(f"Connection failed: {error}")


display(widget, status, table, widgets.HBox([note, add]), shortlist_table)
connection_task = asyncio.create_task(connect())

In [ ]:
shortlist_frame = pd.DataFrame(shortlist.values())
shortlist_frame.to_csv("airway-shortlist.csv", index=False)
shortlist_frame